In [236]:
import torch
from math import sqrt

In [97]:
with open(
    "Obras_completas_ilustradas_Federico_Garcia_Lorca_djvu.txt",
    "r",
    encoding="utf-8"
) as f:

    text = f.read()


In [98]:
chars = sorted(set(text))
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}
encode = lambda s: [stoi[c] for c in s]
decode = lambda l: [itos[i] for i in l]

In [99]:
text[:8]
encoded_text = torch.tensor(encode(text), dtype = torch.long)

In [255]:
train_size = int(len(text)*0.9)
train_data = encoded_text[:train_size]
test_data = encoded_text[train_size:]
block_size = 8
batch_size = 1
torch.manual_seed(0)

def get_batch(split):
    # generate a small batch of data of inputs x and targets y

    ix = torch.randint(len(split) - block_size, (batch_size,))
    x = torch.stack([split[i:i+block_size] for i in ix])
    y = torch.stack([split[i+1:i+block_size+1] for i in ix])
    return x, y

In [256]:
first_batch = get_batch(train_data)
first_batch[1]

tensor([[36, 47, 28,  1,  1, 46, 28, 41]])

In [257]:
vocab_size = len(chars)
d_model = 10
embedding = torch.nn.Embedding(vocab_size, d_model)

In [ ]:
class Attention(torch.nn.Module):

    def __init__(self):
        super().__init__()
        self.

In [309]:
d_k = 5

#class head(torch.nn.Module):

class decoder_only_transformer(torch.nn.Module):

    def __init__(self):
        super().__init__()
        self.embedding = torch.nn.Embedding(vocab_size, d_model)
        self.pos_encoding = torch.nn.Embedding(vocab_size, d_model)
        self.W_q = torch.nn.Linear(d_model, d_k)
        self.W_k = torch.nn.Linear(d_model, d_k)
        self.W_v = torch.nn.Linear(d_model, d_model)
        self.ln_1 = torch.nn.LayerNorm(d_model)
        self.ln_2 = torch.nn.LayerNorm(d_model)
        self.W_ff1 = torch.nn.Linear(d_model, 2*d_model)
        self.W_ff2 = torch.nn.Linear(2*d_model, d_model)
        self.linear = torch.nn.Linear(d_model, vocab_size)


    def forward(self, vec_ins):
        X_e = self.embedding(vec_ins)
        X_e = X_e + self.pos_encoding(vec_ins)
        Q = self.W_q(X_e) #(B,T,d_k)
        K = self.W_k(X_e) #(B,T,d_k)
        V = self.W_v(X_e) #(B,T,d_model)
        print(Q.shape, K.shape, V.shape, Q.T.shape)
        QV = Q @ K.transpose(1,2)
        QV_scale = QV/sqrt(d_k) #(B,T,T)
        QV_softmax = torch.nn.functional.softmax(QV_scale, dim = 2) #(B,T,T)
        added_weights = QV_softmax @ V
        norm_res_att = self.ln_1(X_e + added_weights)
        linear_1 = self.W_ff1(torch.nn.functional.gelu(norm_res_att))
        linear_2 = self.W_ff2(linear_1)
        norm_linear = self.ln_2(linear_2 + norm_res_att)
        logits = self.linear(norm_linear)
        return logits


In [310]:
my_first_model = decoder_only_transformer()
my_first_model(first_batch[0])

torch.Size([1, 8, 5]) torch.Size([1, 8, 5]) torch.Size([1, 8, 10]) torch.Size([5, 8, 1])


tensor([[[-3.8326e-02,  2.1980e-01, -6.3661e-01, -5.2173e-01,  2.3906e-02,
          -2.1391e-01,  1.0809e+00, -1.0469e+00,  2.2955e-01,  3.1199e-01,
          -5.7387e-01,  1.0048e-01, -2.5990e-01, -2.2319e-01,  4.1636e-02,
          -3.3236e-02, -1.7909e-01, -4.9374e-01, -8.5249e-02, -2.7250e-01,
          -7.0304e-02, -6.0656e-01,  2.8298e-01, -1.3924e+00,  1.4854e+00,
          -7.6644e-01,  1.0222e-01, -2.5779e-01, -7.4614e-02,  2.5761e-01,
           3.5117e-01, -3.2188e-02, -2.7648e-01,  2.2654e-01, -7.8227e-02,
          -7.1392e-02,  5.8708e-02,  7.4390e-01, -4.1862e-01,  6.0367e-02,
           9.7024e-01,  2.2924e-01,  1.0552e-01, -3.9636e-01, -4.9300e-02,
          -1.6713e+00,  3.9156e-01, -6.5685e-01, -6.9674e-02,  2.0656e-01,
           5.4367e-02, -2.0170e-01, -1.1575e+00, -8.4671e-01,  3.0029e-01,
          -2.5511e-02, -6.7123e-03,  1.0950e+00, -4.8203e-02, -5.0971e-01,
           4.1227e-01,  6.4806e-01, -6.5309e-01, -9.2880e-02, -2.6702e-01,
          -5.7984e-02, -2

In [319]:
loss_fn = torch.nn.CrossEntropyLoss()
loss = loss_fn(first_batch[0][0], first_batch[1][0])
print(f'Total loss for this batch: {loss.item()}')

RuntimeError: Expected floating point type for target with class probabilities, got Long

In [ ]:
vocab_size = len(chars)
embedding_dim = 10
embedding = torch.nn.Embedding(vocab_size, embedding_dim)